# Goalkeeper first-dive phases and height evidence
Inspect the versioned experiment without replacing confirmed directions, historical height labels or whole-record QA. A silhouette proxy is not a calibrated physical height or a manually verified label.


In [ ]:
from pathlib import Path
import json
from collections import Counter
root = Path.cwd()
if (root / 'release_metrics.json').is_file():
    release = root
else:
    root = next((p for p in [root, *root.parents] if (p / 'src/goalkeeper_dive_phase.py').is_file()), root)
    release = root / 'outputs/goalkeeper_height_phase_2026-10-11/release_v2'
assert release.is_dir(), 'Run and freeze the experiment first; see scripts/run_goalkeeper_height_phase.py'
summary = json.loads((release / 'summary.json').read_text(encoding='utf-8'))
metrics = json.loads((release / 'release_metrics.json').read_text(encoding='utf-8'))
metrics


In [ ]:
# These counts are separate evidence tiers, never model accuracy.
{
    'original_height': dict(Counter(r['dive_height'] for r in summary['rows'])),
    'strict_pose_height_candidate': summary['phase_height_counts'],
    'weaker_silhouette_height_proxy': summary['silhouette_height_proxy_counts'],
    'automatic_whole_record_QA_accepted': sum(r['auto_accepted'] for r in summary['rows']),
}


In [ ]:
clip_id = 'col_eng_p03_cuadrado_col11'
trial = next(r for r in summary['experiments'] if r['clip_id'] == clip_id)
{key: trial['phase_evidence'].get(key) for key in ['reasons', 'tracking_stop_reason', 'frame_window', 'connected_mask_frames', 'source_censored', 'silhouette_phase']}


In [ ]:
from IPython.display import Video, display
video_path = release / trial['review_video']
assert video_path.is_file()
display(Video(filename=str(video_path), embed=True))


## Reproduce
From the project root and the existing project Python environment, run `python -X utf8 scripts/run_goalkeeper_height_phase.py --output-dir outputs/new_phase_run`. Default selection is the 23 original height-unknown clips. Use `--clips` to select the canonical IDs, including known-label controls. Always choose a new output directory.
The source snapshot and hashes record the exact inference settings. SAM2 512 is a CPU experiment, not an assertion of equal quality to the official 1024 setting. Full videos, model weights and upstream caches are local dependencies, not included in the evidence ZIP.
